# Building MCP Servers
In the [Agents Tutorial](5.1-agents-and-tools.ipynb) you saw how LLM models can be combined with tools (Python functions) to make agents which can autonomously search for information or perform actions for you. 

However, sometimes it's better to use an existing agent, rather than have to create a new agent with each new tool. In the last tutorial, we created tools to interact with our InterSystems IRIS database, which we gave to a Langchain agent. Wouldn't it be great if we could give these tools to the agents we already use every day? 

This is where Model Context Protocol (MCP) comes in. MCP is an open protocol for creating and adding new tools to existing agents, and is widely supported among popular agents, including Codex, Claude Code/Desktop, GitHub Copilot, OpenCode, Pi Agent etc. By wrapping tools in an MCP server, they can be used by whatever agent you (or your clients) wish to use. 


## Transport types

The first consideration when creating or using MCP servers should often be the type of transport used, or *"how does the agent program talk to the MCP server"*. There are two main types of transport used, **STDIO** and **Streamable HTTP**. These serve two different purposes: 

### STDIO

Standard Input/Output (STDIO) is used for MCP servers where the tool runs on the same computer as the client application.
- These are used for local tools, like giving an agent tools to use an internet browser or tools to spellcheck a document.
- The execution happens on the same computer as the program calling it. 
- The client launches the server as a subprocess

### Streamable HTTP 

Streamable HTTP is commonly used when the MCP server is provided as a network service.

- The server runs independently and exposes an HTTP endpoint, accessible via a URL.
- Server must be running before a client can connect to it.
- These can provide users access to data on a shared server e.g. exposing internal documentation to employees, searching through remote information etc.
- The execution happens on a different computer as the program calling it (although you can also access local HTTP servers remotely) 
- Deployments should consider authentication, HTTPS and access controls. 

## Available MCP Servers

This tutorial is going to focus on building a Python-based MCP server from scratch to use access data from InterSystems IRIS. However there are a couple of other things you might see referenced elsewhere or find useful for your projects: 

- [**iris-agentic-dev**](https://github.com/intersystems-community/iris-agentic-dev)
    - A pre-defined, open source MCP server for accessing and controlling an InterSystems IRIS instance. It is a superb tool for developing with InterSystems IRIS, particularly for testing and executing ObjectScript scripts.
    - This may be useful for your project if you are building projects directly in InterSystems IRIS. However, this tutorial series has been recommending building client applications that use InterSystems IRIS as a data-store and standard backend, rather than in-depth InterSystems IRIS Development. If you are using IRIS as shown in these tutorials, this MCP server is unlikely to be very useful for you.


- [**InterSystems AI Hub**](https://github.com/intersystems-community/ai-hub-eap/tree/master)
    - A new SDK for building agents, tools, and MCP servers in ObjectScript, currently in Early Access (as of late 2026).
    - Requires a separate InterSystems IRIS build and knowledge of ObjectScript. Not recommended for use in Hackathons at present. 


## Creating MCP Servers for InterSystems IRIS

In the last tutorials, we saw how to give an agent limited access to the InterSystems IRIS instance. Let's do the opposite and build tools that give an agent full access to InterSystems IRIS...

Here, we are going to create an MCP server with `FastMCP` a commonly used Python Library for creating MCP Servers and Clients. If you would rather create MCP tools in other programming languages, there are similar packages for building MCP servers in many different languages. 

Firstly, in case you haven't already install FastMCP. 


In [1]:
#  pip install fastmcp

### Theory

As we have already seen with Langchain, the tools we create for FastMCP servers are simple Python functions. The syntax is slightly different—you need to create the server object, then add a decorator to the function (`@<server>.tool` the line above the function definition `def`), as follows: 

```python
from fastmcp import FastMCP

# Create server
my_mcp = FastMCP()

@my_mcp.tool # decorator to make a tool
def addition_tool(a:int, b:int): 
    """ 
    Tool to add two numbers 
    """
    return a + b
```

After this, we run the function the MCP server with `my_mcp.run(params....)`. The parameters for this run command depend largely on transport method.  

**STDIO Example** 

```python
my_mcp.run() # or my_mcp.run(transport="stdio"), but this is the default
```

With STDIO MCP Servers, configuration required to connect to them is an executable, e.g. 

```json
{ 
    "MyStdioServer" : {
        "command": "python3",
        "args": ["/path/to/my_mcp.py"]
            }
}

```

**HTTP Example**

```python

my_mcp.run(transport="streamable-http",
           port=6789, 
           host="localhost")
```


With HTTP MCP servers, the configuration required to connect resembles a standard HTTP connection. There is a URL and any required headers: 

```json
{
    "MyHTTPServer": { 
        "type":"http",
        "url": "http://localhost:6789/mcp",
        "headers": {"Authorization": "Basic ..."}
    }
}
```

### In Practice

Let's take a look at this in practice. Now MCP servers aren't really designed to be run from interactive notebooks... but we are going to do that anyway, at least with an HTTP Server. 

Here we are going to create a tool that can execute any SQL Queries against our database. To be clear, this is not good practice in production systems (always as limited access scope as possible–a basic security rule for agents and people), but it keeps the code relatively simple for the example.

Let's start by creating our server: 

In [2]:
import iris
from fastmcp import FastMCP

my_mcp = FastMCP()


And our tool. 

Here we connect to InterSystems IRIS in the same way we've seen previously and execute a SQL query given as a tool parameter. There is a basic example of a sanitization to ensure it isn't deleting anything, but again, this is better done at the IRIS permissions level than at the query level. 

In [3]:
@my_mcp.tool
def sql_query(query:str):
    """
    Executes a (read-only) SQL Query against the live InterSystems IRIS database and returns the top 50 results
    """

    # Rudimentary read-only protection for demonstration purposes - more rigorous protection should be applied in prod 
    if "drop" in query.lower():
        return "Query rejected: you may not drop any data."
    
    connection_args = {"hostname":"localhost", "port":32782, "namespace":"FHIRSERVER", "username":"SuperUser", "password": "SYS"}

    conn = iris.connect(**connection_args)
    cursor = conn.cursor() 

    cursor.execute(query)
    # Restrict results to top 50 to limit context window overload
    results = list(cursor.fetchmany(50))

    truncated = cursor.fetchone() is not None
    
    cursor.close()
    conn.close() 
    
    return {"data":results, "row_count":len(results),"truncated": truncated}

To run the server, we just need to run `my_mcp.run()`. Now with this being a run from a Jupyter Notebook, we need to start it within a separate thread which complicates things. It also means that, once started, we cannot easily stop this without restarting the notebook kernel. 

**If you wish to make any changes or stop the server, restart the notebook kernel**

This code is also available as a local file (mcp_server_example_http.py) for alternate use

Run this to start the server, you should see logs appear below: 

In [4]:
import threading

def run_server():
    my_mcp.run(
    transport="streamable-http",
    host="127.0.0.1",
    port=8000,
    show_banner=False
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True,
)
server_thread.start()

### MCP Clients 

MCP clients are often coding agents, so at this point you can connect your coding agent of choice to this server and ask it what it in the database. All you need is the following. The exact syntax depends on the agent, but as an example, for Claude Code you can create the file `.mcp.json` in your project directory with the following: 

```json
{
    "mcpServers":
        {
            "my-iris-server":
                { 
                    "type":"http",
                    "url": "http://localhost:8000/mcp"
                }
        }
}
```

Upon restarting Claude Code, you should see a prompt asking if you want to connect to the MCP server. Similar will be also true for other coding agents, although exact syntax and location of configuration files will vary. 


We can also connect to the MCP Server through custom built Python clients. Let's try connecting and executing a tool. We could do this with the FastMCP Client library, but let's return to using Langchain so it's easy to integrate with the agents we built last time. 

In [5]:
# pip install --upgrade "langchain[mcp]"

In [6]:
from langchain.mcp import MCPAdapter

mcp_client = MCPAdapter("http://localhost:8000/mcp")

async def main():
    async with mcp_client:
        tools = await mcp_client.list_tools()
        print("\nAvailable Tools:\n\n", tools)

        sql_query = next(
            tool for tool in tools
            if tool.name == "sql_query"
        )

        result = await sql_query.ainvoke({
            "query": "SELECT * FROM Sample.Person"
        })

        print("\n\nTest Query: SELECT * FROM Sample.Person\n\nResult:\n")
        print(result)

await main()

[09/28/26 17:32:43] INFO     Starting MCP server 'FastMCP-3786' with transport 'streamable-http'   ]8;id=5977025;file://C:\Users\ging\FHIR-AI-Hackathon-Kit\.venv\Lib\site-packages\fastmcp\server\mixins\transport.py\transport.py]8;;\:]8;id=5977026;file://C:\Users\ging\FHIR-AI-Hackathon-Kit\.venv\Lib\site-packages\fastmcp\server\mixins\transport.py#363\363]8;;\
                             on http://127.0.0.1:8000/mcp                                                          

INFO:     Started server process [10744]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
C:\Users\ging\AppData\Local\Temp\ipykernel_10744\2067855181.py:1: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


INFO:     127.0.0.1:50184 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50184 - "POST /mcp HTTP/1.1" 200 OK

Available Tools:

 [StructuredTool(name='sql_query', description='Executes a (read-only) SQL Query against the live InterSystems IRIS database and returns the top 50 results', args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'query': {'type': 'string'}}, 'required': ['query']}, metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'FastMCP-3786', 'version': '4.0.9'}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x000002799FF47E20>, response_format='content_and_artifact', coroutine=<function as_langchain_tool.<locals>.call_tool at 0x000002799ED268E0>)]
INFO:     127.0.0.1:50184 - "POST /mcp HTTP/1.1" 200 OK


Test Query: SELECT * FROM Sample.Person

Result:

[{'type': 'text', 'text': '{"data":[["Peter","Parker",18],["Bruce","Wayne",50],["Natasha","Romanoff",30],["Clark","Kent",35],["Diana","Prince",32],["Pete

There we have it! You might notice here we are using some asynchronous functions, this is to ensure the  

We can connect to tools over MCP, lets try giving these tools to an agent: 

In [7]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from dotenv import load_dotenv

load_dotenv()


mcp_client = MCPAdapter("http://localhost:8000/mcp")

async def agent_with_tools(prompt:str):
    async with mcp_client:
        tools = await mcp_client.list_tools()

        agent = create_agent(
            model="openai:gpt-5-nano",
            tools=tools,
        )

        return await agent.ainvoke({ # use `ainvoke` because invocation has to be asynchronous for MCP tooling 
            "messages": [
                {"role": "user", "content": prompt}
            ]
        })

In [8]:
response =  await agent_with_tools("What does the HealthCareData table look like? I can't remember what schema it is")
for message in response["messages"]:
    message.pretty_print()

INFO:     127.0.0.1:61050 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:61050 - "POST /mcp HTTP/1.1" 200 OK


[09/28/26 17:33:05] Error calling tool 'sql_query'                                                                 
                    ╭───────────────────────────── Traceback (most recent call last) ─────────────────────────────╮
                    │ C:\Users\ging\FHIR-AI-Hackathon-Kit\.venv\Lib\site-packages\fastmcp\server\server.py:1517   │
                    │ in call_tool                                                                                │
                    │                                                                                             │
                    │ C:\Users\ging\FHIR-AI-Hackathon-Kit\.venv\Lib\site-packages\fastmcp\tools\base.py:440 in    │
                    │ _run                                                                                        │
                    │                                                                                             │
                    │                                   ... 9 frames hidden ...                                   │
                    │                                                                                             │
                    │ in sql_query:16                                                                             │
                    │                                                                                             │
                    │   13 │   conn = iris.connect(**connection_args)                                             │
                    │   14 │   cursor = conn.cursor()                                                             │
                    │   15 │                                                                                      │
                    │ ❱ 16 │   cursor.execute(query)                                                              │
                    │   17 │   # Restrict results to top 50 to limit context window overload                      │
                    │   18 │   results = list(cursor.fetchmany(50))                                               │
                    │   19                                                                                        │
                    │                                                                                             │
                    │ C:\Users\ging\FHIR-AI-Hackathon-Kit\.venv\Lib\site-packages\iris\dbapi\__init__.py:35 in    │
                    │ execute                                                                                     │
                    │                                                                                             │
                    │    32 │   def execute(self, operation: str, parameters: Optional[Union[list, tuple, dict]]  │
                    │       None) -> int:                                                                         │
                    │    33 │   │   """Execute a database operation (query or command)."""                        │
                    │    34 │   │   if parameters is None:                                                        │
                    │ ❱  35 │   │   │   return super().execute(operation)                                         │
                    │    36 │   │   return super().execute(operation, parameters)                                 │
                    │    37 │                                                                                     │
                    │    38 │   def executemany(self, operation: str, seq_of_parameters: Sequence[Sequence]) ->   │
                    │       tuple:                                                                                │
                    ╰─────────────────────────────────────────────────────────────────────────────────────────────╯
                    ProgrammingError: <SQL ERROR>; Details: [SQLCODE: <-51>:<SQL statement expected>]              
                    [Location: <Prepare>]               

INFO:     127.0.0.1:55084 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55085 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:52090 - "POST /mcp HTTP/1.1" 200 OK
================================ Human Message =================================

What does the HealthCareData table look like? I can't remember what schema it is
================================== Ai Message ==================================
Tool Calls:
  sql_query (call_3eSQqIBZE36U958EDfcEk20H)
 Call ID: call_3eSQqIBZE36U958EDfcEk20H
  Args:
    query: DESCRIBE HealthCareData
  sql_query (call_efc15fKouDocW0oSC2SjkWEd)
 Call ID: call_efc15fKouDocW0oSC2SjkWEd
  Args:
    query: SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'HealthCareData' OR TABLE_NAME = 'HEALTHCARADATA'
================================= Tool Message =================================
Name: sql_query

[{'type': 'text', 'text': "Error calling tool 'sql_query': <SQL ERROR>; Details: [SQLCODE: <-51>:<SQL statement expe

Hopefully you should see the agent use multiple tool calls from our MCP server to access information from the InterSystems IRIS database. 

# Summary

Here we have seen how we can create an MCP server with a simple tool to run SQL Queries against the InterSystems IRIS database. We could easily extend this by making additional tools, for example taking the vector search and FHIR query tools from the previous tutorials. 

You can look at these tools being made available from the example files, one with an HTTP Server example (with normal syntax for running that makes up for handling a Jupyter Notebook) and another with a STDIO example. The instructions for using and running these are at the top of the file. 

To formalize and deploy the MCP servers, it is important to make use of the authentication methods built into the protocol to ensure only agents acting on the behalf of authorized users can access the data.

Here, we have deliberately glossed over the security considerations, making use of default admin users. But one of the major advantages to using a data platform like InterSystems IRIS is the built-in security, auditing and user controls that become vital as you take an application from demonstration to deployment. 